# THz Parameter Extraction - Training Notebook

This notebook trains neural networks to extract material parameters (refractive index n, extinction coefficient κ, and thickness d) from THz time-domain spectroscopy data.

**Features:**
- Multiple network architectures (CNN, ResNet, MultiScale, MLP)
- Physics-informed, supervised, or hybrid loss functions
- Automatic model saving and metrics tracking
- Comprehensive evaluation and visualization
- Cloud-ready (Colab, Kaggle, etc.)

## 1. Setup and Installation

Install required packages if running on cloud platforms.

## 2. Imports

In [ ]:
import torch
import torch.optim as optim
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import sys
from pathlib import Path

# Add src to path - works from both notebook dir and project root
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    src_path = notebook_dir.parent / 'src'
else:
    src_path = notebook_dir / 'src'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Import project modules
from training_config import Config
from config_loader import load_training_config, list_available_configs, print_config_summary
from networks import get_network, model_summary
from utils import (
    load_dataset_from_datalake, list_available_datasets,
    THz_Dataset, compute_metrics, print_metrics
)
from train import (
    get_loss_function,
    train_epoch, validate, plot_training_history
)
from evaluate import load_trained_model

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 3. Configuration

Set up training parameters and paths.

In [ ]:
# ============================================================================
# Configuration
# ============================================================================

# Option 1: Load from preset config file (RECOMMENDED)
CONFIG_NAME = 'resnet_base'  # Options: cnn_tiny, cnn_small, cnn_base, cnn_large, resnet_base, resnet_deep

# Uncomment to see available configs
print("Available preset configurations:")
for cfg_name in list_available_configs():
    print(f"  - {cfg_name}")

print(f"\nLoading config: {CONFIG_NAME}")
config = load_training_config(CONFIG_NAME)

# Option 2: Manual configuration (uncomment to use instead)
# from training_config import Config
# config = Config()
# config.DATASET_NAME = 'production_v1'
# config.NUM_EPOCHS = 100
# config.BATCH_SIZE = 64
# config.LEARNING_RATE = 1e-3
# # ... set other parameters

# Print configuration summary
print_config_summary(config)

# Extract network type from config
NETWORK_NAME = config._config_name.split('_')[0] + '_scalable'  # e.g., 'cnn_base' -> 'cnn_scalable'
RUN_NAME = datetime.now().strftime('%Y%m%d_%H%M%S')  # Auto-generated name
DATASET_NAME = 'quick_test'  #config.DATASET_NAME

# Device setup
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"\nRuntime Settings:")
print(f"  Device: {device}")
print(f"  Network: {NETWORK_NAME}")
print(f"  Run name: {RUN_NAME}")

## 4. Load Dataset

Load training and validation data from the datalake.

In [ ]:
# List available datasets
print("Available datasets in datalake:")
available_datasets = list_available_datasets()
for ds in available_datasets:
    marker = " (selected)" if ds == DATASET_NAME else ""
    print(f"  - {ds}{marker}")

if DATASET_NAME not in available_datasets:
    raise ValueError(f"Dataset '{DATASET_NAME}' not found. Available: {available_datasets}")

# Load datasets
print(f"\nLoading dataset: {DATASET_NAME}")
train_data = load_dataset_from_datalake(DATASET_NAME, 'train')
val_data = load_dataset_from_datalake(DATASET_NAME, 'val')

# Create PyTorch datasets
train_dataset = THz_Dataset(train_data['X'], train_data['y'], train_data['T'])
val_dataset = THz_Dataset(val_data['X'], val_data['y'], val_data['T'])

# Create dataloaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=0,  # Set to 0 for notebook compatibility
    pin_memory=True if device.type == 'cuda' else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True if device.type == 'cuda' else False
)

print(f"\n✓ Dataset loaded successfully")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")
print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")

## 5. Initialize Model

Create the neural network and setup training components.

In [ ]:
# Create model
print(f"Initializing {NETWORK_NAME} network...")
model = get_network(NETWORK_NAME, config=config).to(device)
print(f"Model parameters: {model.count_parameters():,}")

# Loss function
criterion = get_loss_function(config)
print(f"Loss function: {config.LOSS_TYPE}")
if config.LOSS_TYPE == 'hybrid':
    print(f"  Alpha: {config.ALPHA}")
    print(f"  Normalized supervised loss: {config.USE_NORMALIZED_SUPERVISED_LOSS}")
    print(f"  Physics loss scale: {config.PHYSICS_LOSS_SCALE}")

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min',
    factor=config.SCHEDULER_FACTOR,
    patience=config.SCHEDULER_PATIENCE
)

print("✓ Model initialized successfully")

In [ ]:
model_text = model_summary(model, False)

print(model_text)

## 6. Setup Save Directories

Create directories for saving models and results.

In [ ]:
# Get save paths from datalake
model_dir = config.get_model_save_path(NETWORK_NAME, RUN_NAME)
results_dir = config.get_results_save_path(NETWORK_NAME, RUN_NAME)

print(f"Save directories:")
print(f"  Models: {model_dir}")
print(f"  Results: {results_dir}")

# Save run configuration
run_config = {
    'network_name': NETWORK_NAME,
    'run_name': RUN_NAME,
    'dataset_name': DATASET_NAME,
    'dataset_metadata': train_data.get('metadata'),
    'config': {k: v for k, v in config.__dict__.items() if not k.startswith('_')},
    'start_time': datetime.now().isoformat()
}

config_file = model_dir / 'run_config.json'
with open(config_file, 'w') as f:
    json.dump(run_config, f, indent=2, default=str)
print(f"\n✓ Saved run config to {config_file}")

## 7. Training Loop

Train the model with early stopping and checkpointing.

In [ ]:
# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'val_metrics': [],
    'learning_rate': []
}

# Training state
best_val_loss = float('inf')
patience_counter = 0

print(f"\n{'='*60}")
print(f"{'Starting Training':^60}")
print(f"{'='*60}\n")

for epoch in range(config.NUM_EPOCHS):
    epoch_start = time.time()
    
    print(f"Epoch {epoch+1}/{config.NUM_EPOCHS}")
    print(f"{'-'*60}")
    
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device, config)
    
    # Validate
    val_loss, val_metrics = validate(model, val_loader, criterion, device, config)
    
    # Update scheduler
    scheduler.step(val_loss)

    # Get current learning rate
    current_lr = optimizer.param_groups[0]['lr']

    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_metrics'].append(val_metrics)
    history['learning_rate'].append(current_lr)
    
    # Print epoch summary
    epoch_time = time.time() - epoch_start
    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.6f}")
    print(f"  Val Loss:   {val_loss:.6f}")
    print(f"  LR:         {current_lr:.2e}")
    print(f"  Time: {epoch_time:.1f}s")
    print(f"  n R²: {val_metrics['n_r2']:.4f} | κ R²: {val_metrics['kappa_r2']:.4f} | d R²: {val_metrics['d_um_r2']:.4f}")
    
    # Save best model
    if val_loss < best_val_loss - config.MIN_DELTA:
        best_val_loss = val_loss
        patience_counter = 0
        
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_metrics': val_metrics,
            'best_val_loss': best_val_loss,
            'config': run_config,
            'network_name': NETWORK_NAME,
            'run_name': RUN_NAME
        }
        
        save_path = model_dir / 'best_model.pt'
        torch.save(checkpoint, save_path)
        print(f"✓ Saved best model (val_loss: {val_loss:.6f})")
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= config.PATIENCE:
        print(f"\nEarly stopping triggered after {epoch+1} epochs")
        print(f"No improvement for {config.PATIENCE} epochs")
        break
    
    print(f"\n{'='*60}\n")

print(f"\n{'='*60}")
print(f"{'Training Complete!':^60}")
print(f"{'='*60}")
print(f"Best validation loss: {best_val_loss:.6f}")

## 8. Save Final Model and History

In [ ]:
# Save final model
final_checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': history['train_loss'][-1],
    'val_loss': history['val_loss'][-1],
    'best_val_loss': best_val_loss,
    'config': run_config,
    'network_name': NETWORK_NAME,
    'run_name': RUN_NAME,
    'history': history
}

save_path = model_dir / 'final_model.pt'
torch.save(final_checkpoint, save_path)
print(f"✓ Saved final model to {save_path}")

# Save training history as JSON
history_path = results_dir / 'training_history.json'
with open(history_path, 'w') as f:
    history_json = {
        'train_loss': history['train_loss'],
        'val_loss': history['val_loss'],
        'val_metrics': [
            {k: float(v) for k, v in m.items()}
            for m in history['val_metrics']
        ],
        'learning_rate': history['learning_rate']
    }
    json.dump(history_json, f, indent=2)
print(f"✓ Saved training history to {history_path}")

## 9. Visualize Training Progress

In [ ]:
# Generate training plots
print("Generating training plots...")
plot_path = results_dir / 'training_plots.png'
fig = plot_training_history(history, save_path=plot_path)
plt.show()

print(f"\n✓ Training plots saved to {plot_path}")

## 10. Model Evaluation

Evaluate the best model on validation data and visualize predictions.

In [ ]:
# Load best model
checkpoint = torch.load(model_dir / 'best_model.pt', map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch']+1}")
print(f"Best validation loss: {checkpoint['best_val_loss']:.6f}")

# Evaluate on validation set
all_preds = []
all_targets = []

with torch.no_grad():
    for batch_data in val_loader:
        X, y = batch_data[:2]
        X = X.to(device)
        
        y_pred = model(X)
        
        all_preds.append(y_pred.cpu())
        all_targets.append(y)

predictions = torch.cat(all_preds, dim=0)
targets = torch.cat(all_targets, dim=0)

# Compute final metrics
final_metrics = compute_metrics(targets, predictions)
print_metrics(final_metrics, title="Final Validation Metrics")

## 11. Prediction Visualizations

In [ ]:
# Plot predicted vs true for each parameter
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

param_names = ['n (Refractive Index)', 'κ (Extinction)', 'd (Thickness, μm)']
units = [1, 1, 1e6]  # Convert d to μm

for i, (ax, name, unit) in enumerate(zip(axes, param_names, units)):
    pred = predictions[:, i].numpy() * unit
    true = targets[:, i].numpy() * unit
    
    # Scatter plot
    ax.scatter(true, pred, alpha=0.3, s=10)
    
    # Perfect prediction line
    min_val, max_val = true.min(), true.max()
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect')
    
    # Get R² from metrics
    if i == 0:
        r2 = final_metrics['n_r2']
    elif i == 1:
        r2 = final_metrics['kappa_r2']
    else:
        r2 = final_metrics['d_um_r2']
    
    ax.set_xlabel(f'True {name}', fontsize=12)
    ax.set_ylabel(f'Predicted {name}', fontsize=12)
    ax.set_title(f'{name}\nR² = {r2:.4f}', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
pred_plot_path = results_dir / 'predictions.png'
plt.savefig(pred_plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved predictions plot to {pred_plot_path}")

## 12. Error Distribution Analysis

In [ ]:
# Plot error distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

param_names = ['n', 'κ', 'd']
units = [1, 1, 1e6]

for i, (ax, name, unit) in enumerate(zip(axes, param_names, units)):
    errors = (predictions[:, i].numpy() - targets[:, i].numpy()) * unit
    
    ax.hist(errors, bins=50, alpha=0.7, edgecolor='black')
    ax.axvline(x=0, color='r', linestyle='--', linewidth=2, label='Zero error')
    ax.axvline(x=np.mean(errors), color='g', linestyle='--', linewidth=2, 
              label=f'Mean: {np.mean(errors):.4f}')
    
    ax.set_xlabel(f'Prediction Error ({name})', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title(f'{name} Error Distribution\nStd: {np.std(errors):.4f}', 
                fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
error_plot_path = results_dir / 'error_distributions.png'
plt.savefig(error_plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved error distribution plot to {error_plot_path}")

## 13. Summary

Print final summary of the training run.

In [ ]:
print(f"\n{'='*70}")
print(f"{'Training Summary':^70}")
print(f"{'='*70}")
print(f"\nRun Information:")
print(f"  Run name: {RUN_NAME}")
print(f"  Network: {NETWORK_NAME}")
print(f"  Dataset: {DATASET_NAME}")
print(f"  Loss type: {config.LOSS_TYPE}")
print(f"  Total epochs: {epoch+1}")
print(f"  Best epoch: {checkpoint['epoch']+1}")

print(f"\nFinal Metrics:")
print(f"  Refractive Index (n):")
print(f"    MAE:  {final_metrics['n_mae']:.5f}")
print(f"    RMSE: {final_metrics['n_rmse']:.5f}")
print(f"    R²:   {final_metrics['n_r2']:.4f}")

print(f"  Extinction Coefficient (κ):")
print(f"    MAE:  {final_metrics['kappa_mae']:.6f}")
print(f"    RMSE: {final_metrics['kappa_rmse']:.6f}")
print(f"    R²:   {final_metrics['kappa_r2']:.4f}")

print(f"  Thickness (d):")
print(f"    MAE:  {final_metrics['d_um_mae']:.2f} μm")
print(f"    RMSE: {final_metrics['d_um_rmse']:.2f} μm")
print(f"    R²:   {final_metrics['d_um_r2']:.4f}")

print(f"\nSaved Files:")
print(f"  Best model: {model_dir / 'best_model.pt'}")
print(f"  Final model: {model_dir / 'final_model.pt'}")
print(f"  Config: {model_dir / 'run_config.json'}")
print(f"  Training history: {results_dir / 'training_history.json'}")
print(f"  Training plots: {results_dir / 'training_plots.png'}")
print(f"  Predictions: {results_dir / 'predictions.png'}")
print(f"  Error distributions: {results_dir / 'error_distributions.png'}")

print(f"\n{'='*70}\n")

## 14. Optional: Load and Test Saved Model

Demonstrate how to load the saved model for inference.

In [ ]:
# Load the best model using the imported function from evaluate.py
# (handles scalable network configs automatically)
loaded_model, loaded_checkpoint = load_trained_model(
    model_dir / 'best_model.pt',
    device=device
)

print("✓ Successfully loaded model for inference")
print(f"  Epoch: {loaded_checkpoint['epoch']+1}")
print(f"  Best val loss: {loaded_checkpoint['best_val_loss']:.6f}")

# Test on a single sample
sample_idx = 0
X_sample = val_dataset[sample_idx][0].unsqueeze(0).to(device)
y_sample = val_dataset[sample_idx][1]

with torch.no_grad():
    prediction = loaded_model(X_sample).cpu().squeeze()

print(f"\nSample prediction:")
print(f"  True: n={y_sample[0]:.3f}, κ={y_sample[1]:.5f}, d={y_sample[2]*1e6:.1f} μm")
print(f"  Pred: n={prediction[0]:.3f}, κ={prediction[1]:.5f}, d={prediction[2]*1e6:.1f} μm")